### Import Libraries

In [19]:
import os
import sys

sys.path.insert(0, os.path.dirname(os.getcwd()))


In [20]:
import torch
from torch import nn

from transformers import WhisperProcessor, WhisperForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model, TrainableTokensModel

from src.model import WhisperAccentConfig, WhisperAccentForConditionalGeneration, WhisperAccentProcessor, register_whisper_accent
from src.train.dataset import DataCollatorSpeechSeq2SeqWithPadding, WhisperDataset
from src.train.train import processor_init, model_init, ModelArguments, LoraArguments, WhisperAccentTrainingArguments
from src.train.trainer import WhisperAccentTrainer
from src.model.tokenization import ACCENTS
from src.utils.loading import load_model_from_pretrained

register_whisper_accent()


### Create Arguments

In [21]:
MODEL_TYPE = "whisper_accent"
BASE_MODEL_NAME_OR_PATH = "openai/whisper-small.en"
IS_MULTILINGUAL = False
DATASET_NAME="westbrook/English_Accent_DataSet"

model_args = ModelArguments(
    model_type=MODEL_TYPE,
    base_model_name_or_path=BASE_MODEL_NAME_OR_PATH,
    is_multilingual=IS_MULTILINGUAL,
)

lora_args = LoraArguments(
    lora_enable=True,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.15,
    lora_bias="none",
    use_rslora=True,
    task_type="SEQ_2_SEQ_LM",
)

training_args = WhisperAccentTrainingArguments(
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    lambda_accent_loss=0.0,
    lambda_diversity_loss=0.0,
    optim="adamw_torch",
    learning_rate=1e-5,
    embedding_learning_rate=5e-5,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_steps=0.05,
    max_steps=100,
    max_grad_norm=1.0,
    eval_strategy="steps",
    eval_steps=50,
    eval_on_start=True,
    predict_with_generate=True,
    logging_first_step=True,
    logging_steps=10,
    remove_unused_columns=False,
    ddp_find_unused_parameters=False,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)


### Load Model and Processor

In [31]:
# Load model and processor; whisper / whisper_accent models are supported
if model_args.model_type == "whisper_accent":
    processor = processor_init(model_args.base_model_name_or_path)
    model = model_init(model_args.base_model_name_or_path, processor)
elif model_args.model_type == "whisper":
    processor = WhisperProcessor.from_pretrained(model_args.base_model_name_or_path)
    model = WhisperForConditionalGeneration.from_pretrained(model_args.base_model_name_or_path)
    # Update generation config; https://github.com/openai/whisper/discussions/2094
    if model.generation_config.is_multilingual:
        model.generation_config.language = "en"
        model.generation_config.task = "transcribe"
    model.generation_config.forced_decoder_ids = None
else:
    raise ValueError(f"Invalid model type: {model_args.model_type}")

# # Add LoRA layers
# # Note: Non-LoRA training is not implemented yet
# if lora_args.lora_enable:
#     # Target linear layers
#     target_modules = []
#     m_list = ["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]
#     for name, _ in model.named_modules():
#         if any(suffix in name for suffix in m_list):
#             target_modules.append(name)

#     lora_config = LoraConfig(
#         r=lora_args.lora_r,
#         lora_alpha=lora_args.lora_alpha,
#         lora_dropout=lora_args.lora_dropout,
#         bias=lora_args.lora_bias,
#         use_rslora=lora_args.use_rslora,
#         # target_modules=target_modules,
#         target_modules=[],
#         task_type=lora_args.task_type,
#         ensure_weight_tying=True,
#     )

#     # Trainable token indices for new accent tokens
#     if model_args.model_type == "whisper_accent":
#         accent_token_indices = sorted(list(model.generation_config.accent_to_id.values()))
#         lora_config.trainable_token_indices = accent_token_indices

#     model = get_peft_model(model, lora_config)
#     model.print_trainable_parameters()
# else:
#     raise NotImplementedError("Non-LoRA training is not implemented yet")


You are using a model of type whisper to instantiate a model of type whisper_accent. This is not supported for all configurations of models and can yield errors.
Loading weights: 100%|██████████| 479/479 [00:01<00:00, 314.88it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]   


In [32]:
from peft import TrainableTokensConfig, get_peft_model

accent_token_indices = sorted(list(model.generation_config.accent_to_id.values()))
config = TrainableTokensConfig(
    token_indices=accent_token_indices,
)

peft_model = get_peft_model(model, config)

In [33]:
peft_model

PeftModel(
  (base_model): TrainableTokensModel(
    (model): WhisperAccentForConditionalGeneration(
      (model): WhisperAccentModel(
        (encoder): WhisperEncoder(
          (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
          (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
          (embed_positions): Embedding(1500, 768)
          (layers): ModuleList(
            (0-11): 12 x WhisperEncoderLayer(
              (self_attn): WhisperAttention(
                (k_proj): Linear(in_features=768, out_features=768, bias=False)
                (v_proj): Linear(in_features=768, out_features=768, bias=True)
                (q_proj): Linear(in_features=768, out_features=768, bias=True)
                (out_proj): Linear(in_features=768, out_features=768, bias=True)
              )
              (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
              (activation_fn): GELUActivation()
              (fc1):

In [24]:
for name, param in peft_model.named_parameters():
    if param.requires_grad:
        print(name)


model.model.decoder.embed_tokens.trainable_tokens_delta.accent_tokens


In [46]:
def custom_token_init(model, n_tokens):
    # Get the top n_tokens indices with the highest magnitude
    embedding = model.get_decoder().embed_tokens
    new_embedding = embedding.weight.data[-1]
    indices = torch.topk(new_embedding.abs(), n_tokens-1).indices

    for i, idx in enumerate(indices):
        embedding.weight.data[-i-1, idx] = 0

In [47]:
custom_token_init(model, 22)

In [51]:
model.get_decoder().embed_tokens.weight.data[-23:].sum(1)

tensor([7.8904, 7.8905, 7.8650, 7.8647, 7.8645, 7.8643, 7.8644, 7.8638, 7.8636,
        7.8625, 7.8606, 7.8573, 7.8555, 7.8552, 7.8549, 7.8547, 7.8539, 7.8536,
        7.8528, 7.8519, 7.8499, 7.8446, 8.2525])

In [ ]:
embedding.data

tensor([ 9.9555e-03,  1.1979e-02,  1.5335e-02,  1.8614e-02,  1.6999e-02,
         6.0230e-03,  2.9823e-03,  2.0652e-03,  4.7898e-03,  6.6271e-03,
         1.4498e-02,  1.1288e-02,  1.6372e-02,  5.6020e-03,  7.5712e-03,
         1.0052e-02,  1.3929e-02,  3.1832e-03,  1.9771e-02,  9.5008e-03,
         1.1319e-02,  1.7779e-02,  1.4570e-02,  1.4155e-02,  6.1564e-03,
         8.2030e-03,  3.5498e-03,  6.9976e-04,  4.5651e-03,  1.7110e-02,
         4.0593e-02,  3.6579e-02,  1.0299e-02,  1.2622e-02,  1.0151e-02,
         4.9738e-03,  1.8496e-03,  2.0655e-02,  1.8037e-02,  4.8318e-03,
         8.7283e-03,  5.4607e-03,  1.5951e-02,  6.7264e-03,  2.3078e-02,
         6.7151e-03,  1.2903e-02,  1.0677e-02,  9.4054e-03,  5.5526e-03,
         1.6155e-02,  7.7874e-03,  1.4863e-02,  7.0076e-03,  4.0874e-03,
         1.1898e-02,  5.3527e-03,  7.1918e-03,  2.5514e-02,  1.2702e-02,
         1.1219e-02,  1.2547e-02,  1.1090e-02,  3.3194e-03,  1.0624e-02,
         1.0176e-02,  9.6127e-03,  1.2188e-02,  1.5

In [ ]:
torch.argwhere()

TypeError: argwhere() takes 1 positional argument but 3 were given

In [21]:
torch.argwhere(dim_mag_order==4)

tensor([[638]])

In [ ]:
embedding.abs()

tensor([9.9555e-03, 1.1979e-02, 1.5335e-02, 1.8614e-02, 1.6999e-02, 6.0230e-03,
        2.9823e-03, 2.0652e-03, 4.7898e-03, 6.6271e-03, 1.4498e-02, 1.1288e-02,
        1.6372e-02, 5.6020e-03, 7.5712e-03, 1.0052e-02, 1.3929e-02, 3.1832e-03,
        1.9771e-02, 9.5008e-03, 1.1319e-02, 1.7779e-02, 1.4570e-02, 1.4155e-02,
        6.1564e-03, 8.2030e-03, 3.5498e-03, 6.9976e-04, 4.5651e-03, 1.7110e-02,
        4.0593e-02, 3.6579e-02, 1.0299e-02, 1.2622e-02, 1.0151e-02, 4.9738e-03,
        1.8496e-03, 2.0655e-02, 1.8037e-02, 4.8318e-03, 8.7283e-03, 5.4607e-03,
        1.5951e-02, 6.7264e-03, 2.3078e-02, 6.7151e-03, 1.2903e-02, 1.0677e-02,
        9.4054e-03, 5.5526e-03, 1.6155e-02, 7.7874e-03, 1.4863e-02, 7.0076e-03,
        4.0874e-03, 1.1898e-02, 5.3527e-03, 7.1918e-03, 2.5514e-02, 1.2702e-02,
        1.1219e-02, 1.2547e-02, 1.1090e-02, 3.3194e-03, 1.0624e-02, 1.0176e-02,
        9.6127e-03, 1.2188e-02, 1.5348e-02, 2.5092e-03, 3.7693e-03, 1.0169e-02,
        9.5330e-03, 2.6140e-02, 1.0343e-

### Training

In [ ]:
import datetime

run_id = "whisper-test-run" 
run_name = f"{run_id}-{datetime.datetime.now().strftime('%Y%m%d-%H%M')}"
output_dir = f"/workspace/checkpoints/{run_id}"


In [ ]:
training_args.set_save(
    strategy="steps",
    steps=50,
    total_limit=100,
)
training_args.set_push_to_hub(
    model_id=f"mavleo96/{run_id}",
    strategy="all_checkpoints",
)
training_args.output_dir = output_dir


In [ ]:
collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

train_dataset = WhisperDataset(
    data_path=DATASET_NAME,
    split="train",
    processor=processor,
    multilingual_model=model_args.is_multilingual,
    num_proc=16,
)

eval_dataset = WhisperDataset(
    data_path=DATASET_NAME,
    split="validation",
    processor=processor,
    multilingual_model=model_args.is_multilingual,
    num_proc=16,
)
eval_dataset.raw_dataset = eval_dataset.raw_dataset.select(range(20))


In [ ]:
trainer = WhisperAccentTrainer(
    model=model,
    args=training_args,
    data_collator=collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=processor,
    compute_metrics="all" if model_args.model_type == "whisper_accent" else "wer",
)

In [ ]:
trainer.train()
